<h1>MVP de Engenharia de Dados</h1>

### Ciência de Dados e Analytics

**Aluno(a):** Jéssica Alves de Melo Caetano  
**Matrícula:** 4052025002261  
**Disciplina:** Engenharia de Dados  



## 1. Contexto e Objetivo do MVP

A produção de café possui relevância econômica e agrícola no Brasil e apresenta diferenças entre estados, regiões, períodos e tipos de café produzidos. A análise desses dados exige que informações provenientes de fontes públicas sejam coletadas, organizadas, tratadas e disponibilizadas de forma estruturada para permitir sua exploração analítica.

Este MVP de Engenharia de Dados tem como objetivo construir um pipeline de dados utilizando o Databricks, tendo como fonte dados da Produção Agrícola Municipal (PAM) do IBGE referentes à produção de café no Brasil entre 2015 e 2025.

O pipeline será estruturado segundo a arquitetura em camadas **Bronze, Silver e Gold**, permitindo preservar os dados ingeridos, realizar seu tratamento e disponibilizar estruturas preparadas para consumo analítico.

A construção do pipeline será orientada pelas seguintes perguntas de negócio:

1. Quais estados e regiões concentram a maior produção de café?
2. Como a produção de café evoluiu ao longo do período analisado?
3. Quais estados apresentam maior produtividade?
4. Como a participação dos principais estados produtores mudou ao longo dos anos?
5. Como a produção de café arábica e conilon se distribui entre os principais estados produtores?

Essas perguntas orientam as etapas de busca, coleta, transformação, modelagem e disponibilização dos dados, permitindo que a construção do pipeline esteja diretamente relacionada aos objetivos analíticos definidos para o projeto.

## 2. Busca pelos Dados

Para o desenvolvimento deste MVP foi utilizada uma base de dados pública do **Instituto Brasileiro de Geografia e Estatística (IBGE)**, proveniente da pesquisa **Produção Agrícola Municipal (PAM)** e disponibilizada por meio do **Sistema IBGE de Recuperação Automática (SIDRA)**.

A escolha dessa fonte ocorreu por sua relação direta com o objetivo definido para o projeto. A base disponibiliza informações sobre a produção agrícola brasileira e permite analisar diferentes indicadores relacionados à cultura do café ao longo do tempo e entre as Unidades da Federação.

Para este MVP, foi utilizado um recorte dos dados referentes à produção de café no Brasil no período de **2015 a 2025**, contendo informações como:

- Unidade da Federação;
- Ano;
- Produto;
- Área destinada à colheita;
- Área colhida;
- Quantidade produzida;
- Rendimento médio da produção;
- Valor da produção.

A presença dessas variáveis permite investigar tanto a evolução temporal da produção quanto diferenças entre estados, regiões e tipos de café, estando, portanto, alinhada às perguntas de negócio definidas para o projeto.

### Fonte e condições de utilização dos dados

**Fonte:** Instituto Brasileiro de Geografia e Estatística (IBGE) — Produção Agrícola Municipal (PAM), disponibilizada por meio do Sistema IBGE de Recuperação Automática (SIDRA).

Os dados utilizados neste MVP são provenientes de uma fonte pública oficial. O IBGE mantém um Plano de Dados Abertos em conformidade com a Política de Dados Abertos do Poder Executivo Federal, instituída pelo Decreto nº 8.777/2016.

Na documentação oficial consultada para este projeto, não foi identificada uma licença Creative Commons específica associada à tabela da PAM utilizada. Por esse motivo, o trabalho não atribui aos dados uma licença específica não declarada pela fonte. O conjunto é documentado como dado público disponibilizado pelo IBGE/SIDRA, mantendo-se a identificação e a atribuição da fonte.

**Período utilizado:** 2015 a 2025.  
**Formato utilizado no projeto:** CSV.  
**Órgão responsável:** Instituto Brasileiro de Geografia e Estatística (IBGE).  
**Pesquisa:** Produção Agrícola Municipal (PAM).  
**Sistema de consulta:** SIDRA.

## 3. Coleta dos Dados

Após a definição da fonte de dados, o arquivo utilizado no projeto foi disponibilizado em formato **CSV** no ambiente Databricks, permitindo que os dados fossem armazenados e processados diretamente na plataforma.

O arquivo foi armazenado em um **Volume do Unity Catalog**, no seguinte caminho:

`/Volumes/workspace/bronze/dados_brutos/ibge_pam_cafe_2015_2025.csv`

A utilização do Volume permite manter o arquivo de origem disponível no ambiente do Databricks e separado das tabelas posteriormente criadas durante o processamento dos dados.

A ingestão foi realizada no notebook **01_ingestão_bronze**, responsável pela leitura do arquivo CSV e pela criação da tabela:

`workspace.bronze.ibge_pam_cafe_raw`

Nesta etapa, buscou-se preservar os dados em uma forma próxima à estrutura original da fonte. A tabela Bronze resultante contém **204 registros** e os campos:

- **linha_original:** conteúdo textual de cada linha do arquivo de origem;
- **data_ingestao:** data e horário em que o registro foi incorporado à camada Bronze.

Essa estratégia mantém uma cópia dos dados ingeridos antes das etapas de tratamento e estruturação, possibilitando rastreabilidade entre o arquivo de origem e as transformações realizadas posteriormente.

Dessa forma, o fluxo inicial de coleta e ingestão do projeto pode ser representado como:

**IBGE/PAM → arquivo CSV → Volume do Unity Catalog → camada Bronze**

## 4. Modelagem dos Dados

A modelagem adotada neste MVP foi estruturada segundo a arquitetura **Medallion**, organizando os dados nas camadas **Bronze, Silver e Gold** dentro do ambiente Lakehouse do Databricks.

Essa organização foi escolhida para separar as diferentes responsabilidades do pipeline e manter a rastreabilidade dos dados desde sua ingestão até a disponibilização para consumo analítico.

### Camada Bronze

A camada Bronze é responsável por armazenar os dados ingeridos em uma estrutura próxima à fonte original.

Nesta camada foi criada a tabela:

`workspace.bronze.ibge_pam_cafe_raw`

O conteúdo original do arquivo CSV foi preservado, juntamente com a informação do momento da ingestão, permitindo que as etapas posteriores de transformação possam ser reproduzidas a partir dos dados brutos.

### Camada Silver

A camada Silver contém os dados tratados e estruturados para utilização nas análises.

Nesta camada foi criada a tabela:

`workspace.silver.ibge_pam_cafe`

Durante a transformação, a estrutura original do arquivo foi interpretada e reorganizada, permitindo representar os dados por ano, produto, Unidade da Federação, variável e valor. Também foram considerados os símbolos e convenções existentes na fonte original do IBGE durante o tratamento dos valores.

A tabela Silver funciona como a principal base detalhada e estruturada do projeto, sendo utilizada como origem para a construção das estruturas analíticas da camada Gold.

### Camada Gold

A camada Gold contém dados refinados e preparados para responder às perguntas analíticas definidas para o MVP.

Foram materializadas as seguintes tabelas:

- `workspace.gold.producao_anual_cafe`: produção total de café por ano;
- `workspace.gold.principais_estados_produtores`: cinco estados com maior produção acumulada no período analisado;
- `workspace.gold.participacao_anual_estados`: produção e participação percentual anual dos principais estados produtores.

Dessa forma, o fluxo de dados implementado no projeto segue a estrutura:

**Arquivo CSV → Bronze → Silver → Gold → Consumo analítico**

A separação em camadas reduz a necessidade de utilizar diretamente os dados brutos nas análises e permite manter diferentes níveis de tratamento e agregação dentro do pipeline.

## 5. Catálogo de Dados

O catálogo de dados documenta as tabelas persistidas nas camadas Bronze, Silver e Gold do projeto. Para cada estrutura são apresentados seu contexto, campos, tipos de dados, domínio e linhagem, permitindo compreender a origem e as transformações realizadas ao longo do pipeline.

### 5.1 Camada Bronze — `workspace.bronze.ibge_pam_cafe_raw`

**Contexto:** tabela responsável por armazenar os dados ingeridos a partir do arquivo CSV da Produção Agrícola Municipal (PAM) do IBGE. Nesta camada, o conteúdo é mantido em estrutura próxima à fonte original, antes das transformações realizadas na camada Silver.

**Quantidade de registros:** 204.

| Campo | Tipo | Descrição | Domínio |
|---|---|---|---|
| `linha_original` | string | Conteúdo textual original de cada linha do arquivo CSV do IBGE. | Texto proveniente diretamente das linhas do arquivo de origem. |
| `data_ingestao` | timestamp | Data e horário em que o registro foi incorporado à camada Bronze. | Data e hora da execução da ingestão. |

**Linhagem:**  
`IBGE/PAM → ibge_pam_cafe_2015_2025.csv → Volume do Unity Catalog → workspace.bronze.ibge_pam_cafe_raw`

Nesta etapa não são realizadas agregações analíticas. A finalidade da tabela é preservar os dados ingeridos e fornecer a origem para as transformações realizadas posteriormente na camada Silver.

### 5.2 Camada Silver — `workspace.silver.ibge_pam_cafe`

**Contexto:** tabela estruturada e tratada a partir dos dados armazenados na camada Bronze. Nesta etapa, as linhas do arquivo original são interpretadas e reorganizadas em registros contendo ano, produto, Unidade da Federação, valor e variável analisada, formando a base detalhada utilizada nas análises do projeto.

**Quantidade de registros:** 6.160.

| Campo | Tipo | Descrição | Domínio |
|---|---|---|---|
| `ano` | integer | Ano de referência da informação agrícola. | 2015 a 2025. |
| `produto` | string | Categoria de produto disponibilizada na fonte do IBGE. | Total; Café (em grão) Total; Café (em grão) Arábica; Café (em grão) Canephora. |
| `uf` | string | Unidade da Federação associada ao registro. | 28 valores presentes na fonte, incluindo as UFs brasileiras e o registro histórico `Guanabara`. |
| `valor` | double | Valor numérico tratado correspondente à combinação de ano, produto, UF e variável. A unidade de medida depende da variável representada no registro. | 0,0 a 56.643.071,0 nos dados utilizados. |
| `valor_original` | string | Valor conforme representado originalmente no arquivo do IBGE, preservado para rastreabilidade e tratamento das convenções existentes na fonte. | Valores numéricos e símbolos/convenções presentes no arquivo de origem. |
| `variavel` | string | Indicador agrícola ao qual o valor do registro se refere. | Área destinada à colheita; Área colhida; Quantidade produzida; Rendimento médio da produção; Valor da produção. |

**Unidades associadas às variáveis:**

- **Área destinada à colheita:** hectares;
- **Área colhida:** hectares;
- **Quantidade produzida:** toneladas;
- **Rendimento médio da produção:** quilogramas por hectare;
- **Valor da produção:** mil reais.

**Observação sobre o domínio de `uf`:** a tabela Silver preserva o valor `Guanabara` por estar presente na estrutura da fonte utilizada. Esse registro não é considerado na preparação da base analítica da camada Gold.

**Linhagem:**  
`workspace.bronze.ibge_pam_cafe_raw → interpretação e estruturação das linhas do CSV → tratamento dos valores → workspace.silver.ibge_pam_cafe`

A camada Silver mantém maior granularidade que as tabelas Gold e funciona como a principal fonte estruturada para as agregações e análises posteriores.

### 5.3 Camada Gold — `workspace.gold.producao_anual_cafe`

**Contexto:** tabela analítica que apresenta a produção total de café no Brasil para cada ano do período analisado. Foi construída a partir dos dados tratados da camada Silver e permite acompanhar a evolução anual da produção nacional de café.

**Quantidade de registros:** 11.

| Campo | Tipo | Descrição | Domínio |
|---|---|---|---|
| `ano` | integer | Ano de referência da produção de café. | 2015 a 2025. |
| `producao_total_toneladas` | double | Quantidade total de café produzida no Brasil no respectivo ano, expressa em toneladas. | 2.647.504,0 a 3.705.719,0 toneladas nos dados analisados. |

**Transformações realizadas:**

Para a construção desta tabela foram considerados os registros referentes à variável **Quantidade produzida**, utilizando a produção total de café. Os dados foram agregados por ano para disponibilizar a produção nacional anual em toneladas.

O registro histórico `Guanabara`, preservado na camada Silver por fazer parte da fonte original, não é considerado na base utilizada para as análises da camada Gold.

**Linhagem:**  
`workspace.silver.ibge_pam_cafe → seleção dos dados de produção de café → exclusão de Guanabara da base analítica → agregação da produção por ano → workspace.gold.producao_anual_cafe`

**Utilização analítica:** esta tabela auxilia principalmente na resposta à pergunta de negócio:

> **Como a produção de café evoluiu ao longo do período analisado?**

### 5.4 Camada Gold — `workspace.gold.principais_estados_produtores`

**Contexto:** tabela analítica que identifica os cinco estados com maior produção acumulada de café no período de 2015 a 2025. Além da Unidade da Federação, a tabela apresenta a região geográfica correspondente e o volume total produzido no período.

**Quantidade de registros:** 5.

| Campo | Tipo | Descrição | Domínio |
|---|---|---|---|
| `uf` | string | Unidade da Federação pertencente ao grupo dos cinco maiores produtores de café no período analisado. | Bahia; Espírito Santo; Minas Gerais; Rondônia; São Paulo. |
| `regiao` | string | Região geográfica brasileira correspondente à Unidade da Federação. | Nordeste; Norte; Sudeste. |
| `producao_total_toneladas` | double | Produção acumulada de café do estado entre 2015 e 2025, expressa em toneladas. | 1.567.415,0 a 17.835.941,0 toneladas nos dados analisados. |

**Transformações realizadas:**

Para a construção desta tabela foram considerados os registros referentes à variável **Quantidade produzida**, utilizando a produção total de café. A produção foi agregada por Unidade da Federação ao longo do período analisado, associada à respectiva região geográfica e ordenada de forma decrescente. Em seguida, foram selecionados os cinco estados com maior produção acumulada.

Os cinco estados resultantes foram **Minas Gerais, Espírito Santo, São Paulo, Bahia e Rondônia**.

**Linhagem:**  
`workspace.silver.ibge_pam_cafe → seleção dos dados de produção de café → exclusão de Guanabara da base analítica → associação das UFs às regiões → agregação da produção por estado → seleção dos cinco maiores produtores → workspace.gold.principais_estados_produtores`

**Utilização analítica:** esta tabela auxilia principalmente na resposta à pergunta de negócio:

> **Quais estados e regiões concentram a maior produção de café?**

### 5.5 Camada Gold — `workspace.gold.participacao_anual_estados`

**Contexto:** tabela analítica que apresenta, para cada ano entre 2015 e 2025, a produção dos cinco principais estados produtores de café e sua participação percentual em relação à produção nacional daquele ano.

**Quantidade de registros:** 55, correspondentes a 11 anos de análise para os 5 principais estados produtores.

| Campo | Tipo | Descrição | Domínio |
|---|---|---|---|
| `ano` | integer | Ano de referência da produção e da participação percentual. | 2015 a 2025. |
| `uf` | string | Unidade da Federação pertencente ao grupo dos cinco principais produtores de café. | Bahia; Espírito Santo; Minas Gerais; Rondônia; São Paulo. |
| `producao_estado_toneladas` | double | Quantidade de café produzida pelo estado no respectivo ano, expressa em toneladas. | 84.734,0 a 2.064.689,0 toneladas nos dados analisados. |
| `producao_total_toneladas` | double | Produção total nacional de café no respectivo ano, utilizada como referência para o cálculo da participação. | 2.647.504,0 a 3.705.719,0 toneladas nos dados analisados. |
| `participacao_percentual` | double | Participação percentual da produção do estado em relação à produção nacional no respectivo ano. | 2,99% a 60,66% nos dados analisados. |

**Transformações realizadas:**

A tabela foi construída a partir da produção anual dos cinco estados identificados como os principais produtores no período analisado. Para cada combinação de ano e estado, a produção estadual foi comparada à produção total nacional do mesmo ano.

A participação percentual foi calculada pela relação:

**Participação percentual = (Produção do estado / Produção total nacional) × 100**

O resultado foi arredondado para duas casas decimais.

**Linhagem:**  
`workspace.silver.ibge_pam_cafe → seleção dos dados de produção de café → exclusão de Guanabara da base analítica → identificação dos cinco principais estados produtores → agregação anual por estado → cálculo da produção nacional anual → cálculo da participação percentual → workspace.gold.participacao_anual_estados`

**Utilização analítica:** esta tabela auxilia principalmente na resposta à pergunta de negócio:

> **Como a participação dos principais estados produtores mudou ao longo dos anos?**

## 6. Arquitetura e Linhagem do Pipeline

O fluxo de dados desenvolvido neste MVP segue a arquitetura em camadas **Bronze, Silver e Gold**, permitindo acompanhar os dados desde sua origem até as estruturas preparadas para consumo analítico.

### Fluxo geral dos dados

**IBGE — Produção Agrícola Municipal (PAM)**  
↓  
**Arquivo CSV — `ibge_pam_cafe_2015_2025.csv`**  
↓  
**Volume do Unity Catalog — Dados Brutos**  
↓  
**BRONZE — `workspace.bronze.ibge_pam_cafe_raw`**  
↓  
*Ingestão e preservação dos dados de origem*  
↓  
**SILVER — `workspace.silver.ibge_pam_cafe`**  
↓  
*Interpretação, tratamento e estruturação dos dados*  
↓  
**GOLD — Estruturas para consumo analítico**

↳ `workspace.gold.producao_anual_cafe`  
↳ `workspace.gold.principais_estados_produtores`  
↳ `workspace.gold.participacao_anual_estados`

### Responsabilidade de cada camada

**Bronze:** preservação dos dados ingeridos em estrutura próxima à fonte original.

**Silver:** tratamento, interpretação e estruturação dos dados, mantendo uma base detalhada para utilização nas análises.

**Gold:** agregação e disponibilização de estruturas orientadas às perguntas de negócio e ao consumo analítico.

## 7. Referências

- **IBGE — Instituto Brasileiro de Geografia e Estatística.** Produção Agrícola Municipal (PAM). Sistema IBGE de Recuperação Automática (SIDRA).  
  Disponível em: https://sidra.ibge.gov.br/pesquisa/pam/tabelas

- **IBGE — Instituto Brasileiro de Geografia e Estatística.** Dados Abertos.  
  Disponível em: https://www.ibge.gov.br/acesso-informacao/dados-abertos.html

- **Databricks.** Documentação oficial — Unity Catalog.  
  Disponível em: https://docs.databricks.com/aws/en/data-governance/unity-catalog/

- **Databricks.** Documentação oficial — Medallion Architecture.  
  Disponível em: https://docs.databricks.com/aws/en/lakehouse/medallion.html